Using the file CompiledCANONInfo.txt, use the OpenAI API to train a chatbot that can answer questions about Star Wars


In [2]:
import openai
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import UnstructuredFileLoader
from langchain_openai import OpenAIEmbeddings
from langchain.chains.question_answering import load_qa_chain
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import GPT4AllEmbeddings, HuggingFaceEmbeddings 
from langchain_community.llms import GPT4All
from langchain.chains import RetrievalQA

In [3]:
# Load the .env file
load_dotenv()
# Load the OpenAI API key from the .env file
openai.api_key = os.getenv("OPENAI_API_KEY")
# Load the .txt file using the UnstructuredTextLoader
print("Loading data...")
loader = UnstructuredFileLoader("CompiledALLInfo.txt")
data = loader.load()
print("Data loaded.")

persist_directory = "vectDBALLINFOHuggingFace"
# embedding = GPT4AllEmbeddings()#OpenAIEmbeddings() 
embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
def generateVectorDB(data):
    print("Splitting document...")
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
    documents = text_splitter.split_documents(data)
    print("Number of documents: ", len(documents))
    print("Documents split.")
    """TODO: GET AN EMBEDDING THAT'LL LET ME USE THE VECTOR STORES"""
    print("Generating vector database...")
    db = Chroma.from_documents(documents, embedding, persist_directory=persist_directory)
    db.persist()
    print("Vector database created.")
generateVectorDB(data)
# print(docs[0].page_content)

Loading data...
Data loaded.


c:\Users\zs811\AppData\Local\Programs\Python\Python39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
modules.json: 100%|██████████| 349/349 [00:00<00:00, 99.2kB/s]
c:\Users\zs811\AppData\Local\Programs\Python\Python39\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\zs811\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on

Splitting document...
Number of documents:  950938
Documents split.
Generating vector database...
Vector database created.


In [4]:
# Now, we can use the QA chain to answer questions
# TODO: Setup database retrieval to then use to query and generate an answer
def generateRetriever(allInfo=False):
    if allInfo:
        persist_directory = "vectDBALLINFOHuggingFace"
    else:
        persist_directory = "vectDB"
    embedding = GPT4AllEmbeddings()
    db = Chroma(persist_directory=persist_directory,embedding_function=embedding)
    retriever = db.as_retriever(search_type="similarity")
    print("Retriever created.")
    return retriever

def generateQAChain(retriever=None):
    myLLM = GPT4All(model="models/mistral-7b-openorca.Q4_0.gguf", n_threads=8)
    print("LLM created.")
    qa = RetrievalQA.from_chain_type(llm=myLLM, chain_type="map_reduce", retriever=retriever, return_source_documents=True)
    print("QA chain created.")
    return qa

qa = generateQAChain(retriever=generateRetriever(allInfo=True))


Retriever created.
LLM created.
QA chain created.


In [8]:
query = "Who is captain rex"
result = qa({"query": query})
print(result)

{'query': 'Who is captain rex', 'result': ' Captain Rex is a character from Star Wars, specifically a clone trooper who went on to become a commander before shedding his designation after Order 66 was issued.', 'source_documents': [Document(page_content='Rex wearing his Phase II clone trooper armor during the Clone WarsA legend amongst his clone brothers,[26] Rex was a brave Clone Captain,[27] and later a Clone Commander, until he shed his designation after Order 66. Rex was produced from the template of the bounty hunter Jango Fett. [5] As with all of his fellow clone troopers, Rex lived to serve the Galactic Republic[15] and to protect its citizens. [24] While he was eager to follow orders, Rex adopted a less rigid approach as the Clone Wars wore on. [16] During the Clone Wars, Rex would work alongside the Clone Commander Cody,[29] the Jedi Generals Obi-Wan Kenobi and Anakin Skywalker, and the Jedi Padawan Ahsoka Tano. [5][84] He was also loyal to his men, feeling that protecting the